## PROYECTO 2:ESTUDIO DE SEÑALES EEG DEL FUNCIONAMIENTO Y EL CONTROL DEL MOVIMIENTO (imaginería motora) CON ENFOQUE EN EVALUACIÓN DE SISTEMAS DE INTERFAZ CEREBRO-COMPUTADORA (BCI)
### Dataset: EEG Motor Movement/Imagery – PhysioNet / OpenNeuro ds004362
#### **Autoras:** [Luisa Fernanda Llamas Baldovino, Camila Andrea Montiel Zapata]
#### **Laboratorio de Bioseñales – UdeA 2026-1**

> **Guía de lectura:** Este notebook implementa cada etapa del flujo de procesamiento de señales para BCI de forma independiente y paso a paso.


In [ ]:
# ─── Importación de librerías ───────────────────────────────────────
import os
import glob
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from scipy import signal as sp_signal
from scipy.signal import firwin, filtfilt, hilbert
from scipy.stats import shapiro, wilcoxon, ttest_rel

import mne
mne.set_log_level('WARNING')



In [7]:

# ─── Parámetros globales ─────────────────────────────────────────────
SFREQ           = 160          # Hz
CANALES_INTERES = ['C3', 'Cz', 'C4']
L_FREQ, H_FREQ  = 8.0, 30.0   # Banda motora Mu + Beta
MU_BAND         = (8, 13)      # Hz
BETA_BAND       = (13, 30)     # Hz
TMIN, TMAX      = 0.0, 4.0     # Ventana de época (segundos)
M_WELCH         = 128          # Muestras por segmento Welch
S_WELCH         = M_WELCH // 2

# ─── Etiquetas de clases ─────────────────────────────────────────────
NOMBRES_CLASE = {0: 'Reposo (T0)', 1: 'Img Izq (T1)', 2: 'Img Der (T2)'}
COLORES_CLASE = {0: '#7F8C8D', 1: '#2980B9', 2: '#E74C3C'}


In [8]:
# ════════════════════════════════════════════════════════════════════
# ÍNDICE 1: PSD (Densidad Espectral de Potencia) - Welch
# ════════════════════════════════════════════════════════════════════
def calcular_psd_banda(senal_1d, fs=SFREQ, banda=(8, 13)):
    """
    Calcula la potencia media en una banda de frecuencia usando Welch.
    Retorna un escalar: potencia promedio en µV².
    """
    f, Pxx = sp_signal.welch(senal_1d, fs=fs, nperseg=M_WELCH, noverlap=S_WELCH)
    mask = (f >= banda[0]) & (f <= banda[1])
    return np.mean(Pxx[mask])


## **1.Creación  de la función de lectura de señal EEG filtrada y permita obtener PSD e indices de clasificación.**

In [9]:
# ─── AJUSTAR ESTA RUTA AL DIRECTORIO LOCAL DEL DATASET ──────────────
RUTA_BASE = r"C:\Users\HP\Desktop\PROYECTO_EEG_BCI_1\eeg-motor-movementimagery-dataset-1.0.0\files"

# ─── Selección de sujetos ────────────────────────────────────────────
# Para el análisis completo vamos a  usar todos los sujetos: S001–S109
# Para prueba rápida, limitar a pocos sujetos
N_SUJETOS = 109   # Cambiar a un número menor para pruebas rápidas (ej. 10)

todos_sujetos = [f"S{i:03d}" for i in range(1, N_SUJETOS + 1)]

# ─── Convención de archivos PhysioNet ────────────────────────────────
# Runs de IMAGINACIÓN MOTORA (manos): R04, R08, R12
# IMPORTANTE: T1 = puño izquierdo, T2 = puño derecho (en AMBOS tipos de run)

RUNS_IMAGINACION = ["R04", "R08", "R12"]

def get_tipo_run(nombre_archivo):
    """Devuelve el tipo de condición según el nombre del archivo."""
    for r in RUNS_IMAGINACION:
        if r in nombre_archivo:
            return "Imaginacion_Motora"
    return "Desconocido"

# ─── Buscar archivos de imaginación para todos los sujetos ───────────
archivos_img = []
for sujeto in todos_sujetos:
    for run in RUNS_IMAGINACION:
        patron = os.path.join(RUTA_BASE, sujeto, f"{sujeto}{run}.edf")
        if os.path.exists(patron):
            archivos_img.append(patron)

archivos_img.sort()
print(f"Archivos de IMAGINACIÓN encontrados: {len(archivos_img)}")
print(f"  Esperados: {N_SUJETOS * len(RUNS_IMAGINACION)} (máximo)")
print(f"\nPrimeros 5 archivos:")
for a in archivos_img[:5]:
    print(f"  {os.path.basename(a)} → {get_tipo_run(a)}")

Archivos de IMAGINACIÓN encontrados: 327
  Esperados: 327 (máximo)

Primeros 5 archivos:
  S001R04.edf → Imaginacion_Motora
  S001R08.edf → Imaginacion_Motora
  S001R12.edf → Imaginacion_Motora
  S002R04.edf → Imaginacion_Motora
  S002R08.edf → Imaginacion_Motora


In [3]:
#carga del archivos EEG